# 🚴 RideAware BA2 – BERT Fine-Tuning

**Schritte:**
1. GPU aktivieren: `Laufzeit → Laufzeittyp ändern → T4 GPU`
2. Pakete installieren (Zelle 1)
3. `train_split.csv` und `test.csv` hochladen (Zelle 3)
4. Alle Zellen ausführen
5. Ergebnisse & Modell herunterladen

⏱️ Dauer mit GPU: ca. **3–5 Minuten**

## Zelle 1 – Pakete installieren

In [ ]:
!pip install transformers torch scikit-learn pandas -q
print('✅ Pakete installiert!')

## Zelle 2 – GPU prüfen

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU verfügbar: {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
else:
    print('⚠️  Keine GPU – bitte Laufzeittyp auf T4 GPU ändern!')
    print('   Laufzeit → Laufzeittyp ändern → T4 GPU → Speichern')
    device = torch.device('cpu')

print(f'Gerät: {device}')

## Zelle 3 – Daten hochladen

Lade deine `train_split.csv` und `test.csv` hoch.

In [ ]:
from google.colab import files
import pandas as pd
import io

print('Bitte train_split.csv und test.csv hochladen...')
uploaded = files.upload()

df_train = pd.read_csv(io.BytesIO(uploaded['train_split.csv']))
df_test  = pd.read_csv(io.BytesIO(uploaded['test.csv']))

df_train['text']  = df_train['text'].astype(str)
df_train['label'] = df_train['label'].astype(str)
df_test['text']   = df_test['text'].astype(str)
df_test['label']  = df_test['label'].astype(str)

print(f'\n✅ Training: {len(df_train)} Beispiele')
print(f'✅ Test:     {len(df_test)} Beispiele')
print('\nKlassenverteilung Training:')
print(df_train['label'].value_counts())

## Zelle 4 – Konfiguration

In [ ]:
# ── Hier kannst du die Einstellungen anpassen ──────────────────────────────

MODEL_NAME = 'deepset/gbert-base'   # Deutsches BERT

# Kategorien – anpassen sobald Georg bestätigt!
LABELS = [
    'Gefahrenstelle',
    'Hindernis',
    'Markierung oder Schild',
    'Ampel',
    'Lückenschluss',
]

EPOCHS     = 3       # Anzahl Trainingsdurchläufe (3 reicht für BA)
BATCH_SIZE = 16      # Größe der Trainings-Batches
LR         = 2e-5    # Learning Rate
MAX_LEN    = 128     # Max. Tokenlänge (kurze Texte → 128 reicht)

LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

print('✅ Konfiguration gesetzt')
print(f'   Modell:     {MODEL_NAME}')
print(f'   Kategorien: {LABELS}')
print(f'   Epochen:    {EPOCHS}')

## Zelle 5 – Tokenizer & Modell laden

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print(f'Lade {MODEL_NAME}...')
print('(Beim ersten Mal wird das Modell heruntergeladen ~440MB)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'\n✅ Modell geladen')
print(f'   Parameter: {total_params:,}')

## Zelle 6 – Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ReportDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


# Labels zu IDs konvertieren
# Zeilen mit unbekannten Labels rausfiltern
df_train = df_train[df_train['label'].isin(LABELS)].reset_index(drop=True)
df_test  = df_test[df_test['label'].isin(LABELS)].reset_index(drop=True)

X_train = df_train['text'].tolist()
y_train = [LABEL2ID[l] for l in df_train['label']]
X_test  = df_test['text'].tolist()
y_test  = [LABEL2ID[l] for l in df_test['label']]

train_dataset = ReportDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset  = ReportDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f'✅ Dataset erstellt')
print(f'   Train Batches: {len(train_loader)}')
print(f'   Test Batches:  {len(test_loader)}')

## Zelle 7 – Training

In [ ]:
import time
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

history = []
best_f1    = 0.0
best_state = None
start_total = time.time()

print(f'Training startet ({EPOCHS} Epochen)...\n')

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    start_epoch = time.time()

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)

    # ── Evaluate ─────────────────────────────────────────────────────────
    model.eval()
    all_preds, all_labels_eval = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels_eval.extend(batch['labels'].numpy())

    acc = accuracy_score(all_labels_eval, all_preds)
    f1m = f1_score(all_labels_eval, all_preds, average='macro', zero_division=0)
    elapsed = time.time() - start_epoch

    print(f'Epoche {epoch}/{EPOCHS}  |  Loss: {avg_loss:.4f}  |  Acc: {acc:.3f}  |  F1: {f1m:.3f}  |  {elapsed:.0f}s')
    history.append({'epoch': epoch, 'loss': avg_loss, 'accuracy': acc, 'f1_macro': f1m})

    if f1m > best_f1:
        best_f1    = f1m
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

total_time = time.time() - start_total
print(f'\n✅ Training fertig! Gesamtzeit: {total_time:.0f}s ({total_time/60:.1f} min)')
print(f'   Bester F1-Score: {best_f1:.3f}')

## Zelle 8 – Finales Ergebnis

In [ ]:
from sklearn.metrics import classification_report

# Bestes Modell laden
if best_state:
    model.load_state_dict(best_state)

model.eval()
all_preds, all_labels_final = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels_final.extend(batch['labels'].numpy())

label_names = [ID2LABEL[i] for i in all_labels_final]
pred_names  = [ID2LABEL[i] for i in all_preds]

acc_final = accuracy_score(label_names, pred_names)
f1_final  = f1_score(label_names, pred_names, average='macro', zero_division=0)

print('=' * 60)
print('ERGEBNIS AUF TESTSET')
print('=' * 60)
print(f'Accuracy:  {acc_final:.3f}')
print(f'Macro-F1:  {f1_final:.3f}\n')
print(classification_report(label_names, pred_names, labels=LABELS, zero_division=0))

## Zelle 9 – Lernkurve visualisieren

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('BERT Trainingsverlauf', fontsize=14, fontweight='bold')

axes[0].plot(hist_df['epoch'], hist_df['loss'],     marker='o', color='#ff5a5f')
axes[0].set_title('Loss');     axes[0].set_xlabel('Epoche')

axes[1].plot(hist_df['epoch'], hist_df['accuracy'], marker='o', color='#4f8cff')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoche'); axes[1].set_ylim(0, 1)

axes[2].plot(hist_df['epoch'], hist_df['f1_macro'], marker='o', color='#2dd4bf')
axes[2].set_title('Macro-F1'); axes[2].set_xlabel('Epoche'); axes[2].set_ylim(0, 1)

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('bert_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik gespeichert: bert_training_curve.png')

## Zelle 10 – Ergebnisse & Modell herunterladen

In [ ]:
import os
import zipfile

# Trainingshistorie speichern
hist_df.to_csv('bert_training_history.csv', index=False)

# Modell speichern
model.save_pretrained('bert_model')
tokenizer.save_pretrained('bert_model')

# Alles in ZIP packen
with zipfile.ZipFile('rideaware_bert_results.zip', 'w') as zf:
    zf.write('bert_training_history.csv')
    zf.write('bert_training_curve.png')
    for root, dirs, files in os.walk('bert_model'):
        for file in files:
            zf.write(os.path.join(root, file))

print('✅ ZIP erstellt: rideaware_bert_results.zip')

# Herunterladen
files.download('rideaware_bert_results.zip')
files.download('bert_training_curve.png')
print('✅ Download gestartet!')

## Zelle 11 – Schnelltest (optional)

Teste das Modell mit eigenen Texten!

In [ ]:
def predict(texts):
    model.eval()
    encodings = tokenizer(
        texts, truncation=True, padding=True,
        max_length=128, return_tensors='pt'
    )
    with torch.no_grad():
        input_ids      = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)
        preds   = torch.argmax(probs, dim=1)
    return [(ID2LABEL[preds[i].item()], probs[i][preds[i]].item()) for i in range(len(texts))]


# ── Hier kannst du eigene Texte testen ────────────────────────────────────
test_texts = [
    'Schlagloch auf dem Radweg beim Praterstern',
    'Gefährliche Kreuzung ohne Sicht',
    'Ampel für Radfahrer defekt',
    'Radwegmarkierung fehlt komplett',
    'Radweg endet plötzlich ohne Anschluss',
]

print('Vorhersagen:\n')
results = predict(test_texts)
for text, (label, conf) in zip(test_texts, results):
    print(f'  {conf:.2f}  {label:<25}  {text}')